# Laboratory: Constructing a Goal-Based Agent Using a Large Language Model

**Course:** Artificial Intelligence — Agents Laboratory  
**Subject:** Goal-Based Agent Architecture, Problem Formulation, State-Space Search (BFS vs. A*), and LLM-Assisted Software Engineering.

---

## 1. Executive Summary & Conceptual Foundations

In classical Artificial Intelligence (Russell & Norvig), an **agent** is an entity that perceives its environment through **sensors** and acts upon that environment through **actuators**.

### Agent Taxonomy: Why Simple Reflex Agents Fail at Navigation
1. **Simple Reflex Agent:** Selects actions strictly based on the current percept using condition-action rules (`if condition then action`). Such agents possess no internal memory of past states and cannot evaluate future consequences. In environments with obstacles, walls, or dead ends, a simple reflex agent is susceptible to infinite loops and deadlocks.
2. **Model-Based Reflex Agent:** Maintains an internal state tracking aspects of the environment that cannot be viewed currently, but still lacks an explicit target destination.
3. **Goal-Based Agent:** Combines an internal model of the world with an **explicit objective (goal)**. The agent simulates or plans ahead: *"What will happen if I take action A? Does that bring me closer to achieving my goal?"* Action selection is driven by search or planning algorithms.
4. **Utility-Based Agent:** Evaluates trade-offs when multiple paths exist using a continuous utility function (e.g., balancing speed, battery consumption, and safety margins).

```
+-----------------------------------------------------------------------+
|                          GOAL-BASED AGENT                             |
|                                                                       |
|   Sensors ----> State Estimation ----> [Current State (r, c)]         |
|                                                |                      |
|   Transition Model: "What action A does"       |                      |
|   World Layout:     "Obstacles / Free space"   v                      |
|                                       [Search / Planner] <--- [Goal]  |
|                                                |                      |
|   Actuators <--------------------------- Action Selection             |
+-----------------------------------------------------------------------+
```

### Formal Problem Formulation (State-Space Search)
- **State Space ($S$):** All grid coordinates $(r, c) \in \{0, \dots, 6\} \times \{0, \dots, 20\}$ where cell $(r, c) \neq \text{'#'}$.
- **Initial State ($s_0$):** Loading bay position $S = (1, 1)$.
- **Actions ($A(s)$):** Cardinal transitions $\{\text{Up}, \text{Down}, \text{Left}, \text{Right}\}$, constrained by walls and boundaries.
- **Transition Model ($T(s, a)$):** Deterministic movement yielding successor $s' = (r + \Delta r, c + \Delta c)$.
- **Goal Test ($G(s)$):** Returns true if and only if $s = (1, 19)$.
- **Path Cost ($c(s, a, s')$):** Uniform unit cost ($1$ move per step).

In [ ]:
from collections import deque
import heapq
import time
from typing import List, Tuple, Dict, Optional, Set
import matplotlib.pyplot as plt
import numpy as np

# Verify Python environment
print("Environment initialized successfully.")

## Task 1: Understanding the Problem

### 1. What is the environment?
The environment is a **2D discrete grid world** with dimensions **7 rows by 21 columns** (147 total grid cells).  
Under Russell & Norvig's environmental properties:
- **Fully Observable:** The agent has complete access to the static warehouse map, obstacle positions, and start/goal coordinates.
- **Deterministic:** Each action reliably advances the vehicle by exactly one grid square without noise or slip.
- **Static:** The layout, obstacles, and destination do not change during planning or execution.
- **Discrete:** Coordinates, actions, and time steps are discrete integers.
- **Single-Agent:** The vehicle is the only decision-making entity.
- **Sequential:** Current action choices directly constrain and determine future reachable states and cumulative path cost.

### 2. What is the goal of the agent?
The explicit objective is to navigate the autonomous vehicle from the initial loading bay position **$S = (1, 1)$** to the target dispatch area **$G = (1, 19)$** along a collision-free path that minimizes total movement cost (number of steps).

### 3. What actions are available to the agent?
The agent possesses four discrete directional actions:
$$\mathcal{A} = \{\text{Up } (-1, 0), \, \text{Down } (+1, 0), \, \text{Left } (0, -1), \, \text{Right } (0, +1)\}$$
An action is valid if and only if the resulting destination coordinates lie within the grid boundaries and do not collide with an obstacle ('#').

### 4. What information must the agent maintain in order to choose its next action?
- **Environmental Information:** The grid dimensions, static obstacle locations, and goal coordinates $(r_G, c_G)$.
- **Internal State:** The agent's current position $(r, c)$.
- **Search/Planning Data Structures:**
  1. **Frontier (Open Set):** Nodes discovered but not yet expanded, ordered by path cost $g(n)$ or evaluation function $f(n) = g(n) + h(n)$.
  2. **Explored Set (Closed List):** States already visited to prevent redundant expansion and infinite loops.
  3. **Parent Pointers / Path History:** Back-pointers to reconstruct the optimal sequence of actions upon reaching $G$.

### 5. Why is this an example of a goal-based agent rather than a simple reflex agent?
A **simple reflex agent** chooses actions based purely on immediate sensory inputs via pre-programmed condition-action rules (`if front_clear then move_forward else turn_right`). In a warehouse with concave obstacle clusters and dead ends, a simple reflex agent cannot recognize whether an action moves it closer to the final destination, resulting in cyclic oscillations or entrapment in dead ends.

In contrast, a **goal-based agent** maintains an explicit representation of the desired goal state ($G$) and combines it with a transition model of its actions. It plans ahead by simulating sequences of future actions through state-space search to evaluate whether an action sequence achieves the goal before executing the first physical move.

### Think About It (Task 1)
> **Prompt:** *Suppose the warehouse becomes twice as large. Would the same search strategy still be appropriate? What additional difficulties might arise?*

**In-Depth Analysis:**
1. **Combinatorial State Space Explosion:**  
   Doubling both dimensions increases the area fourfold ($2W \times 2H = 4 \times \text{Area}$). For uninformed search like **Breadth-First Search (BFS)**, time and memory complexities scale as $\mathcal{O}(b^d)$, where branching factor $b \le 4$ and solution depth $d$ approximately doubles. BFS rapidly becomes intractable due to exponential memory consumption.
2. **Informed Heuristic Search (A*):**  
   **A\* Search with the Manhattan Distance heuristic** remains mathematically appropriate because it focuses search along the gradient towards the goal, expanding far fewer nodes than BFS.
3. **Emergent Scalability Challenges:**
   - **Memory Bottlenecks:** A* stores all open and closed nodes in memory. In massive industrial warehouses, memory exhaustion necessitates **Memory-Bounded A\* (SMA\*)**, **Iterative Deepening A\* (IDA\*)**, or **Hierarchical Pathfinding (HPA\*)**.
   - **Heuristic Depressions / Concave Shelving:** Large warehouse shelving units create long, dead-end aisles. In these regions, Manhattan distance remains small even when deep detours are required, causing heuristic search to temporarily revert to exhaustive breadth-first exploration.
   - **Dynamic Obstacles:** In real-world warehouses, other vehicles and workers move unpredictably. Static offline search must be replaced with real-time replanning algorithms such as **D\* Lite**, **LPA\***, or **Time-Elastic Bands (TEB)**.

## Task 2: Designing the Agent

### Component Identification
- **Environment:** 2D grid world containing passable aisles and impassable shelving units.
- **Current State:** Coordinate tuple $(r, c) \in \mathbb{N}^2$.
- **Goal:** Coordinate tuple $(1, 19)$ corresponding to the dispatch area.
- **Available Actions:** Directional translations: $\{\text{Up}, \text{Down}, \text{Left}, \text{Right}\}$.
- **Decision-Making Component:** An A* search engine using an admissible and consistent Manhattan distance heuristic $h(n) = |r - r_G| + |c - c_G|$ that plans a sequence of actions before execution.

In [ ]:
class WarehouseEnvironment:
    """Represents the 2D grid warehouse environment."""
    DEFAULT_MAP = [
        "#####################",
        "#S....#............G#",
        "#.##....##########..#",
        "#....##.............#",
        "#.######.###.#.###..#",
        "#........#..........#",
        "#####################"
    ]

    ACTIONS = {
        'Up': (-1, 0),
        'Down': (1, 0),
        'Left': (0, -1),
        'Right': (0, 1)
    }

    def __init__(self, map_layout: Optional[List[str]] = None):
        self.grid = map_layout if map_layout is not None else self.DEFAULT_MAP
        self.rows = len(self.grid)
        self.cols = len(self.grid[0])
        self.start = self._find_char('S')
        self.goal = self._find_char('G')
        self.obstacles = {(r, c) for r in range(self.rows) for c in range(self.cols) if self.grid[r][c] == '#'}

    def _find_char(self, char: str) -> Tuple[int, int]:
        for r in range(self.rows):
            for c in range(self.cols):
                if self.grid[r][c] == char:
                    return (r, c)
        raise ValueError(f"Character {char} not found in map.")

    def is_valid(self, state: Tuple[int, int]) -> bool:
        r, c = state
        return 0 <= r < self.rows and 0 <= c < self.cols and state not in self.obstacles

    def get_successors(self, state: Tuple[int, int]) -> List[Tuple[str, Tuple[int, int]]]:
        r, c = state
        successors = []
        for action, (dr, dc) in self.ACTIONS.items():
            nxt = (r + dr, c + dc)
            if self.is_valid(nxt):
                successors.append((action, nxt))
        return successors

    def render(self, path: Optional[List[Tuple[int, int]]] = None) -> str:
        path_set = set(path) if path else set()
        lines = []
        for r in range(self.rows):
            row_chars = []
            for c in range(self.cols):
                pos = (r, c)
                if pos == self.start: row_chars.append('S')
                elif pos == self.goal: row_chars.append('G')
                elif pos in path_set: row_chars.append('*')
                elif pos in self.obstacles: row_chars.append('#')
                else: row_chars.append('.')
            lines.append("".join(row_chars))
        return "\n".join(lines)

env = WarehouseEnvironment()
print(f"Warehouse Grid: {env.rows} rows x {env.cols} columns")
print(f"Start Position: {env.start}, Goal Position: {env.goal}")
print(env.render())

## Task 3: Prompt Engineering with LLMs

### 1. Prompt Specification
We supplied the following detailed specification prompt to the LLM assistant:
> *"Write a well-documented Python program implementing a goal-based agent for the warehouse navigation problem shown above. The program should: represent the warehouse as a two-dimensional grid; determine a collision-free path from S to G; avoid all obstacles; print either the path found or a suitable message if no path exists; explain the search algorithm that has been chosen and why it is appropriate."*

### 2. Evaluation & Analysis of LLM-Generated Output

#### Question 1: Did the LLM generate a working program on the first attempt?
**Yes.** The generated program was syntactically correct and successfully parsed the 2D grid, located `S` and `G`, executed search, and extracted the collision-free trajectory. The modular structure clearly separated the grid model from the search algorithm.

#### Question 2: If not, how can you improve your prompt?
While the first-attempt code worked, several prompt refinements improved engineering robustness:
1. **Specify tie-breaking behavior:** In priority queues, passing tuple `(f, state)` can raise a `TypeError` if `f`-values are equal and coordinates cannot be compared directly. Adding a counter tie-breaker `(f, count, state)` ensures numerical safety.
2. **Specify path visualization:** Prompting the LLM to provide an ASCII grid overlay with `*` characters makes verification immediate.
3. **Request edge case handling:** Explicitly requesting a test with an unreachable goal verifies that the agent gracefully handles deadlock scenarios without infinite loops.

#### Question 3: What search algorithm did the LLM choose?
The LLM selected **A\* Search (A-Star)** using the **Manhattan Distance heuristic** ($h(n) = |r_n - r_G| + |c_n - c_G|$).

#### Question 4: Why do you think the LLM selected this algorithm?
- **Optimality:** In a 4-connected grid with uniform step cost ($c = 1$), Manhattan distance never overestimates the true remaining distance ($h(n) \le h^*(n)$). This **admissibility** guarantees finding the shortest path.
- **Efficiency:** By directing exploration towards the goal coordinate, A* dramatically prunes the search space compared to uninformed algorithms like BFS or Dijkstra.
- **Standard Benchmark Solution:** Grid pathfinding with obstacle avoidance is canonically solved via A* in AI textbooks, making it the highest-probability, best-practice pattern in the LLM's training corpus.

In [ ]:
class GoalBasedAgent:
    """Autonomous goal-based navigation agent supporting BFS and A* search."""
    def __init__(self, env: WarehouseEnvironment, algorithm: str = 'astar'):
        self.env = env
        self.algorithm = algorithm.lower()

    @staticmethod
    def manhattan_distance(p1: Tuple[int, int], p2: Tuple[int, int]) -> int:
        return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

    def plan_path(self) -> Dict[str, any]:
        start_time = time.perf_counter()
        if self.algorithm == 'bfs':
            res = self._bfs()
        elif self.algorithm in ('astar', 'a*'):
            res = self._astar()
        else:
            raise ValueError(f"Unknown algorithm: {self.algorithm}")
        res['time_ms'] = (time.perf_counter() - start_time) * 1000
        return res

    def _bfs(self) -> Dict[str, any]:
        start, goal = self.env.start, self.env.goal
        queue = deque([(start, [start], [])])
        visited = {start}
        nodes_expanded = 0

        while queue:
            curr, path, acts = queue.popleft()
            nodes_expanded += 1
            if curr == goal:
                return {'found': True, 'path': path, 'actions': acts, 'nodes_expanded': nodes_expanded, 'cost': len(acts)}
            for act, nxt in self.env.get_successors(curr):
                if nxt not in visited:
                    visited.add(nxt)
                    queue.append((nxt, path + [nxt], acts + [act]))
        return {'found': False, 'path': [], 'actions': [], 'nodes_expanded': nodes_expanded, 'cost': 0}

    def _astar(self) -> Dict[str, any]:
        start, goal = self.env.start, self.env.goal
        counter = 0
        pq = [(self.manhattan_distance(start, goal), counter, 0, start, [start], [])]
        g_costs = {start: 0}
        nodes_expanded = 0

        while pq:
            f, _, g, curr, path, acts = heapq.heappop(pq)
            nodes_expanded += 1
            if curr == goal:
                return {'found': True, 'path': path, 'actions': acts, 'nodes_expanded': nodes_expanded, 'cost': len(acts)}
            if g > g_costs.get(curr, float('inf')):
                continue
            for act, nxt in self.env.get_successors(curr):
                tentative_g = g + 1
                if tentative_g < g_costs.get(nxt, float('inf')):
                    g_costs[nxt] = tentative_g
                    counter += 1
                    f_score = tentative_g + self.manhattan_distance(nxt, goal)
                    heapq.heappush(pq, (f_score, counter, tentative_g, nxt, path + [nxt], acts + [act]))
        return {'found': False, 'path': [], 'actions': [], 'nodes_expanded': nodes_expanded, 'cost': 0}

In [ ]:
# Execute and Compare Search Algorithms
astar_agent = GoalBasedAgent(env, 'astar')
astar_res = astar_agent.plan_path()

bfs_agent = GoalBasedAgent(env, 'bfs')
bfs_res = bfs_agent.plan_path()

print("=== A* SEARCH RESULTS ===")
print(f"Found: {astar_res['found']} | Steps: {astar_res['cost']} | Nodes Expanded: {astar_res['nodes_expanded']} | Time: {astar_res['time_ms']:.3f} ms")
print("Action Sequence:", " -> ".join(astar_res['actions']))

print("\n=== BFS SEARCH RESULTS ===")
print(f"Found: {bfs_res['found']} | Steps: {bfs_res['cost']} | Nodes Expanded: {bfs_res['nodes_expanded']} | Time: {bfs_res['time_ms']:.3f} ms")

print("\n=== VISUALIZED PATH OVERLAY (* = vehicle path) ===")
print(env.render(astar_res['path']))

### Edge-Case Testing: Unreachable Destination
We verify that the agent gracefully handles a situation where shelving units completely wall off the dispatch area `G`.

In [ ]:
# Create wall-blocked scenario around Goal (1, 19)
blocked_layout = [row for row in env.DEFAULT_MAP]
r1 = list(blocked_layout[1]); r1[18] = '#'; blocked_layout[1] = "".join(r1)
r2 = list(blocked_layout[2]); r2[19] = '#'; blocked_layout[2] = "".join(r2)

blocked_env = WarehouseEnvironment(blocked_layout)
blocked_agent = GoalBasedAgent(blocked_env, 'astar')
blocked_res = blocked_agent.plan_path()

print("Blocked Goal Test Result:")
print(f"Path Found: {blocked_res['found']}")
print(f"Nodes Explored Before Terminating: {blocked_res['nodes_expanded']}")
if not blocked_res['found']:
    print("SUCCESS: Agent correctly recognized that no collision-free path exists.")

## Graphical Visualization of Navigation Solution
Below we render the 2D grid layout, the comparison of explored frontiers, and the final planned trajectory.

In [ ]:
# Matplotlib 2D Grid Plot
fig, ax = plt.subplots(figsize=(10, 3.8), dpi=150)
rows, cols = env.rows, env.cols

for r in range(rows):
    for c in range(cols):
        if (r, c) in env.obstacles:
            ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, facecolor='#2C3E50', edgecolor='gray'))
        else:
            ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, facecolor='#FFFFFF', edgecolor='#BDC3C7'))

# Plot path
path = astar_res['path']
pr = [p[0] for p in path]
pc = [p[1] for p in path]
ax.plot(pc, pr, color='#E67E22', linewidth=3, marker='o', markersize=4, label='A* Planned Trajectory (20 moves)')

# Mark Start and Goal
ax.plot(env.start[1], env.start[0], 's', color='#27AE60', markersize=14, label='Start S (1, 1)')
ax.plot(env.goal[1], env.goal[0], '*', color='#E74C3C', markersize=18, label='Goal G (1, 19)')

ax.set_title('Goal-Based Agent Optimal Trajectory (A* Search)', fontsize=12, weight='bold')
ax.set_xlim(-0.5, cols - 0.5)
ax.set_ylim(rows - 0.5, -0.5)
ax.set_aspect('equal')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.3), ncol=3, fontsize=9)
plt.tight_layout()
plt.show()

## 5. Critical Evaluation of LLM-Assisted Software Engineering

### Strengths
1. **Rapid Prototyping:** The LLM generates boilerplate data structures, grid parsers, and algorithm implementations in seconds.
2. **Algorithm Translation:** Readily translates algorithmic principles (like A* or BFS) into working Python implementations with appropriate libraries (`heapq`, `deque`).
3. **Self-Documentation:** Automatically produces docstrings, type hints, and code comments explaining the design rationale.

### Limitations & Human-in-the-Loop Responsibilities
1. **Edge Case Blind Spots:** Without explicit instructions, LLMs often omit edge case handling (e.g., tie-breaking in priority queues or unreachable goals).
2. **Subtle Off-by-One Errors:** Coordinate systems (`row, col` vs. `x, y`) can be inverted or misaligned unless verified against the grid definition.
3. **Responsibility for Validation:** The human engineer remains responsible for mathematical verification, ensuring that the selected heuristic is truly admissible and consistent, and validating code correctness through empirical testing.